# Parse lines from an ArcGIS Server log

In [1]:
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

In [2]:
# specify the log file
filename = r"C:\Projects\Esri\ArcGIS Server Logs Examination\server-20260118.194808-3600-0.0.log"

In [3]:
# read file contents to a string
text = Path(filename).read_text()

# wrap the lines of the log file (XML) in a root element for processing with ElementTree
wrapped = f"<Root>{text}</Root>"

# create the ElementTree
root = ET.fromstring(wrapped)

In [4]:
type(root)

xml.etree.ElementTree.Element

## get the log entry
- parse the attributes

```
{'time': '2026-01-29T09:05:33,254',
 'type': 'INFO',
 'code': '9999',
 'source': 'Rest',
 'process': '5740',
 'thread': '1',
 'methodName': '',
 'machine': 'SRGWLRSGISSERV.UTAH.UTAD.STATE.UT.US',
 'user': '',
 'elapsed': '4853.0',
 'requestID': 'afc07662-517a-4c3e-b5c7-df4c577dfd6c'}
 ```

- get the body

```Request user: Anonymous user, Service: Public/UDOT_Routes/MapServer```

In [ ]:
# print any lines with this service in the `body`
svc = r"Public/UDOT_Routes"

In [ ]:
# tab delim
print(f"time\ttype\tcode\tsource\telapsed\tbody\trequestID\tArcSOCs")

socs = 0


for msg in root.findall("Msg"):
    attrs = msg.attrib          # dict of key/value pairs
    body = msg.text or ""       # message text

    time = datetime.fromisoformat(attrs['time'])
    type = attrs['type']
    code = attrs['code']
    sourc = attrs['source']
    user = attrs['user']
    elap = 0 if not attrs['elapsed'] else int(float(attrs['elapsed']))
    rqid = attrs['requestID']

    # if svc in body:
        # print(f"{time}  type: {type:<8}  code: {code}   source: {sourc:<30} user:{user:<12} elapsed: {elap:>5}   requestID: {rqid}   {body}")
        # print(f"{str(time):<26}  type: {type:<8}  code: {code}   source: {sourc:<30} elapsed: {elap:>4}   {body}")

    if "created successfully" in body:
        socs += 1
    elif "Shutting down" in body:
        socs -= 1

    arcsocs = f"base +{socs}"

    # tab delimited
    print(f"{str(time)}\t{type}\t{code}\t{sourc}\t{elap}\t{body}\t{rqid}\t{socs:+}")